In [ ]:
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}, GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")

import numpy as np

# ---------------------------------------------------------------------------------------------------------------
# Load Dataset
# ---------------------------------------------------------------------------------------------------------------

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = fetch_california_housing()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"shape of X is  {X.shape}")
print(f"Training data shape  {X_train.shape}")
print(f"Training labels shape  {y_train.shape}")

shape of X is  (20640, 8)
Training data shape  (16512, 8)
Training labels shape  (16512,)


In [3]:
# -----------------------------
# Treat features as sequence
# (each feature = one token)
# -----------------------------
X_train = np.expand_dims(X_train, axis=-1)  # (samples, features, embedding (1))
X_test = np.expand_dims(X_test, axis=-1)

print(f"shape of X_train is  {X_train.shape}")
print(f"shape of Y_train is  {y_train.shape}")

shape of X_train is  (16512, 8, 1)
shape of Y_train is  (16512,)


In [18]:
num_features = X_train.shape[1]
d_model = 32
num_heads = 4
dff = 64

In [27]:
# -----------------------------
# Positional Encoding
# -----------------------------
def positional_encoding(seq_len = num_features, d_model = d_model):
    positions = np.arange(seq_len)[:, np.newaxis]
    dimensions = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / d_model)
    angle_rads = positions * angle_rates

    pos_encoding = np.zeros((seq_len, d_model))
    pos_encoding[:, 0::2] = np.sin(angle_rads[:, 0::2])
    pos_encoding[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(pos_encoding, dtype=tf.float32)


# Multi-head Attention
# -----------------------------
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.dense = tf.keras.layers.Dense(d_model)

    def scaled_dot_product_attention(q, k, v):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], dtype=tf.float32)
        scaled_logits = matmul_qk / tf.math.sqrt(dk)
        weights = tf.nn.softmax(scaled_logits, axis=-1)
        return tf.matmul(weights, v)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, x):
        batch_size = tf.shape(x)[0]

        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)

        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        attention = self.scaled_dot_product_attention(q, k, v)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])

        concat = tf.reshape(attention, (batch_size, -1, self.d_model))
        return self.dense(concat)


# -----------------------------
# Transformer Block
# -----------------------------
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model=d_model, num_heads=4):
        super().__init__()
        # key_dim = d_model // num_heads = 8
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        # self.att = MultiHeadAttention(d_model, num_heads)

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='gelu'),
            tf.keras.layers.Dropout(0.1),
            tf.keras.layers.Dense(d_model),
            tf.keras.layers.Dropout(0.1),
        ])
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout = tf.keras.layers.Dropout(0.1)

    def call(self, x):
        attn = self.att(query=x, key=x, value=x)
        attn = self.dropout(attn)
        x = self.norm1(x + attn)
        ffn_output = self.ffn(x)
        return self.norm2(x + ffn_output)


# -----------------------------
# Model
# -----------------------------
class TransformerRegressor(tf.keras.Model):
    def __init__(self,  d_model=d_model):
        super().__init__()
        self.embedding = tf.keras.layers.Dense(d_model)  # (32,1)
        self.pos_encoding = positional_encoding()  # (8,32)
        self.transformer = TransformerBlock()
        self.pool = tf.keras.layers.GlobalAveragePooling1D()
        self.out = tf.keras.layers.Dense(1)

    def call(self, x):
        x = self.embedding(x)
        x += self.pos_encoding
        x = self.transformer(x)
        x = self.pool(x)
        return self.out(x)


In [28]:
model = TransformerRegressor()
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1)

Epoch 1/10
465/465 [==============================] - 9s 18ms/step - loss: 0.7605 - mae: 0.6543 - val_loss: 0.5868 - val_mae: 0.5957
Epoch 2/10
465/465 [==============================] - 4s 8ms/step - loss: 0.5782 - mae: 0.5608 - val_loss: 0.5635 - val_mae: 0.5944
Epoch 3/10
465/465 [==============================] - 3s 6ms/step - loss: 0.5208 - mae: 0.5249 - val_loss: 0.4592 - val_mae: 0.5200
Epoch 4/10
465/465 [==============================] - 5s 11ms/step - loss: 0.4646 - mae: 0.4913 - val_loss: 0.4315 - val_mae: 0.4969
Epoch 5/10
465/465 [==============================] - 5s 11ms/step - loss: 0.4303 - mae: 0.4691 - val_loss: 0.4887 - val_mae: 0.5595
Epoch 6/10
465/465 [==============================] - 2s 5ms/step - loss: 0.4045 - mae: 0.4552 - val_loss: 0.4483 - val_mae: 0.5228
Epoch 7/10
465/465 [==============================] - 4s 10ms/step - loss: 0.3847 - mae: 0.4429 - val_loss: 0.3472 - val_mae: 0.4096
Epoch 8/10
465/465 [==============================] - 3s 6ms/step - loss

In [29]:
loss, mae = model.evaluate(X_test, y_test)
print("Test MAE:", mae)

preds = model.predict(X_test[:5])
print("Predictions:", preds.flatten())
print("Actual:", y_test[:5])

129/129 [==============================] - 1s 9ms/step - loss: 0.5224 - mae: 0.5667
Test MAE: 0.5666544437408447
1/1 [==============================] - 0s 175ms/step
Predictions: [1.7306443 4.2052355 1.8940934 3.3178585 1.868777 ]
Actual: [1.018 3.648 5.    2.503 1.525]
